# MovieMind – AI Movie Recommendation System
## 1. Project Title and Overview

Google Colab edition | MovieLens latest-small

This notebook adapts the existing MovieMind Flask project for Google Colab. It preserves the project's genre TF-IDF and cosine similarity, preferred-genre ranking, and Bayesian-smoothed popularity ranking. It is a content-based recommender with ratings-aware ranking, not collaborative filtering.

## 2. Problem Statement
Movie catalogs are large, so viewers need a useful way to discover films that match their interests. MovieMind uses movie genres and community ratings to produce explainable recommendations without requiring a production account or neural network.

## 3. Objectives
- Load and clean MovieLens movie and rating records.
- Explore rating, genre, and catalog patterns with charts.
- Reuse MovieMind's existing genre TF-IDF and cosine similarity algorithm.
- Recommend films from a selected movie or preferred genres.
- Evaluate recommendations on a deterministic holdout and show actual outputs.

## 4. Dataset Introduction – MovieLens
The notebook uses MovieLens latest-small from GroupLens. The core files are movies.csv (movie IDs, titles, genres) and ratings.csv (user IDs, movie IDs, ratings, timestamps). The current app does not use optional links.csv or tags.csv. Ratings use a 0.5–5.0 scale and genres are pipe-separated. Dataset statistics and scores below are calculated from the files actually loaded.

## 5. Import Required Libraries
Colab generally includes this data-science stack. The local Flask application's requirements are left unchanged; Matplotlib is used for notebook charts. Uncomment the install command only if your runtime reports a missing package.

In [ ]:
# Optional only if a package is missing:
# %pip -q install pandas numpy matplotlib scikit-learn

import re
import zipfile
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

RANDOM_STATE = 42
pd.set_option("display.max_colwidth", 90)
plt.style.use("seaborn-v0_8-whitegrid")
print("Libraries imported. NumPy:", np.__version__, "| pandas:", pd.__version__)

## 6. Download or Upload Dataset in Google Colab
By default, this downloads and extracts the public latest-small archive to /content. If automatic download fails, the cell asks you to upload movies.csv and ratings.csv from the extracted MovieLens folder. You may also upload both files directly into /content before running this cell. No local Windows paths are used.

In [ ]:
CONTENT_DIR = Path("/content")
DATA_DIR = CONTENT_DIR / "ml-latest-small"
MOVIES_PATH, RATINGS_PATH = CONTENT_DIR / "movies.csv", CONTENT_DIR / "ratings.csv"
ZIP_PATH = CONTENT_DIR / "ml-latest-small.zip"
DATA_URL = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"

if (DATA_DIR / "movies.csv").exists() and (DATA_DIR / "ratings.csv").exists():
    MOVIES_PATH, RATINGS_PATH = DATA_DIR / "movies.csv", DATA_DIR / "ratings.csv"
elif MOVIES_PATH.exists() and RATINGS_PATH.exists():
    pass
else:
    try:
        print("Downloading MovieLens latest-small from GroupLens...")
        urlretrieve(DATA_URL, ZIP_PATH)
        with zipfile.ZipFile(ZIP_PATH) as archive:
            archive.extractall(CONTENT_DIR)
        MOVIES_PATH, RATINGS_PATH = DATA_DIR / "movies.csv", DATA_DIR / "ratings.csv"
    except Exception as download_error:
        print("Automatic download failed:", download_error)
        try:
            from google.colab import files
            print("Upload both movies.csv and ratings.csv.")
            uploaded = files.upload()
            if not {"movies.csv", "ratings.csv"}.issubset(uploaded):
                raise FileNotFoundError("Both CSV files are required.")
            MOVIES_PATH, RATINGS_PATH = CONTENT_DIR / "movies.csv", CONTENT_DIR / "ratings.csv"
        except ImportError as exc:
            raise RuntimeError("Download failed. In Colab, upload both CSVs to /content and rerun.") from exc

if not MOVIES_PATH.exists() or not RATINGS_PATH.exists():
    raise FileNotFoundError("Both movies.csv and ratings.csv are required.")
print("Movies:", MOVIES_PATH, "| Ratings:", RATINGS_PATH)

## 7. Load and Explore the Dataset
Inspect raw dimensions and records. The recommender uses movieId, title, genres, and userId, movieId, rating; timestamps are retained in the loaded table but do not affect ranking.

In [ ]:
movies_raw = pd.read_csv(MOVIES_PATH)
ratings_raw = pd.read_csv(RATINGS_PATH)
needed_movies = {"movieId", "title", "genres"}
needed_ratings = {"userId", "movieId", "rating"}
if not needed_movies.issubset(movies_raw.columns):
    raise ValueError(f"movies.csv needs {sorted(needed_movies)}; got {list(movies_raw.columns)}")
if not needed_ratings.issubset(ratings_raw.columns):
    raise ValueError(f"ratings.csv needs {sorted(needed_ratings)}; got {list(ratings_raw.columns)}")
print(f"Raw movies: {movies_raw.shape[0]:,} rows x {movies_raw.shape[1]} columns")
print(f"Raw ratings: {ratings_raw.shape[0]:,} rows x {ratings_raw.shape[1]} columns")
display(movies_raw.head())
display(ratings_raw.head())

## 8. Data Cleaning and Preprocessing
Following MovieMind's loader: remove duplicate or incomplete movie/user-rating rows, convert ratings to numbers, keep valid 0.5–5.0 ratings, and use an explicit fallback for missing genres. Ratings for movie IDs absent from the catalog are excluded. Removed rows are counted from the actual data.

In [ ]:
raw_movie_rows, raw_rating_rows = len(movies_raw), len(ratings_raw)
movies = movies_raw.drop_duplicates("movieId").dropna(subset=["movieId", "title"]).copy()
ratings = ratings_raw.drop_duplicates(["userId", "movieId"]).dropna(subset=["userId", "movieId", "rating"]).copy()
ratings["rating"] = pd.to_numeric(ratings["rating"], errors="coerce")
ratings = ratings[ratings["rating"].between(0.5, 5.0)].copy()
movies["movieId"] = pd.to_numeric(movies["movieId"], errors="coerce")
ratings["movieId"] = pd.to_numeric(ratings["movieId"], errors="coerce")
ratings["userId"] = pd.to_numeric(ratings["userId"], errors="coerce")
movies = movies.dropna(subset=["movieId"]).copy()
ratings = ratings.dropna(subset=["movieId", "userId"]).copy()
movies["movieId"] = movies["movieId"].astype(int)
ratings["movieId"] = ratings["movieId"].astype(int)
ratings["userId"] = ratings["userId"].astype(int)
movies["title"] = movies["title"].astype(str).str.strip()
movies["genres"] = movies["genres"].fillna("(no genres listed)").astype(str)
ratings["rating"] = ratings["rating"].astype(float)
catalog_ids = set(movies["movieId"])
orphan_count = int((~ratings["movieId"].isin(catalog_ids)).sum())
ratings = ratings[ratings["movieId"].isin(catalog_ids)].copy()

summary = pd.DataFrame({
    "Check": ["Movies loaded", "Movie rows removed", "Ratings loaded",
              "Rating rows removed", "Unknown movie ratings removed",
              "Clean movies", "Clean ratings"],
    "Count": [raw_movie_rows, raw_movie_rows-len(movies), raw_rating_rows,
              raw_rating_rows-len(ratings)-orphan_count, orphan_count, len(movies), len(ratings)]
})
display(summary)
display(pd.DataFrame({
    "movies_missing": movies[["movieId","title","genres"]].isna().sum(),
    "ratings_missing": ratings[["userId","movieId","rating"]].isna().sum()
}))

## 9. Exploratory Data Analysis (EDA) with charts
Visualize rating frequency, common genres, and the most-rated movies. All plots use the loaded CSV records.

In [ ]:
genre_counts = (
    movies.assign(genre=movies["genres"].str.split("|")).explode("genre")
    .query("genre != '(no genres listed)'").groupby("genre")["movieId"].nunique()
    .sort_values(ascending=False)
)
top_counts = ratings.groupby("movieId").size().sort_values(ascending=False).head(10)
top_movies = top_counts.rename("rating_count").reset_index().merge(
    movies[["movieId","title"]], on="movieId", how="left"
)
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
ratings["rating"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color="#2b6f86")
axes[0].set(title="Rating distribution", xlabel="Rating (0.5–5)", ylabel="Number of ratings")
axes[0].tick_params(axis="x", rotation=45)
genre_counts.head(12).sort_values().plot(kind="barh", ax=axes[1], color="#78a9a3")
axes[1].set(title="Most common genres", xlabel="Movies")
top_movies.sort_values("rating_count").plot(
    x="title", y="rating_count", kind="barh", legend=False, ax=axes[2], color="#d07a55"
)
axes[2].set(title="Most-rated movies", xlabel="Number of ratings", ylabel="")
plt.tight_layout()
plt.show()
print(f"Users: {ratings['userId'].nunique():,} | Average rating: {ratings.rating.mean():.2f}/5 | Genres: {len(genre_counts)}")

## 10. Build the Movie Recommendation System
The following class is a notebook adaptation of the existing src/recommender.py. Genre TF-IDF vectors are created once and reused. Ratings supply mean/count statistics; Bayesian shrinkage pulls low-count movie scores toward the global mean.

In [ ]:
class Recommender:
    # MovieMind's content similarity and ratings-aware ranking implementation.
    def __init__(self, movies, ratings):
        self.movies = movies.copy().reset_index(drop=True)
        self.ratings = ratings.copy()
        self.movies["genres"] = self.movies["genres"].fillna("(no genres listed)")
        self.genres = sorted({
            g for value in self.movies.genres for g in value.split("|")
            if g and g != "(no genres listed)"
        })
        self.index = {int(mid): i for i, mid in enumerate(self.movies.movieId)}
        self.genre_positions = {
            g: np.flatnonzero(self.movies.genres.str.split("|").map(lambda values: g in values).to_numpy())
            for g in self.genres
        }
        self.vectorizer = TfidfVectorizer(token_pattern=r"[^|]+")
        self.features = self.vectorizer.fit_transform(self.movies.genres)
        self.stats = self.ratings.groupby("movieId").rating.agg(["mean", "count"])
        self.global_mean = float(self.ratings.rating.mean()) if len(self.ratings) else 3.5
        self.movies["rating"] = self.movies.movieId.map(self.stats["mean"]).fillna(self.global_mean)
        self.movies["rating_count"] = self.movies.movieId.map(self.stats["count"]).fillna(0).astype(int)
        self.movies["score"] = (
            self.movies.rating_count*self.movies.rating + 10*self.global_mean
        )/(self.movies.rating_count+10)

    def _record(self, row, reason="", similarity=None):
        return {"movieId": int(row.movieId), "title": str(row.title),
                "genres": str(row.genres).replace("|", ", "),
                "rating": round(float(row.rating), 2), "rating_count": int(row.rating_count),
                "reason": reason, "similarity": round(float(similarity), 3) if similarity is not None else None}

    def movie(self, movie_id):
        try: movie_id = int(movie_id)
        except (TypeError, ValueError): return None
        if movie_id not in self.index: return None
        return self._record(self.movies.iloc[self.index[movie_id]])

    def popular(self, limit=10):
        limit = max(1, int(limit))
        ranked = self.movies.sort_values(["score","rating_count"], ascending=False).head(limit)
        return [self._record(row, "Popular with MovieLens users") for _,row in ranked.iterrows()]

    def similar(self, movie_id, limit=10):
        try: movie_id = int(movie_id)
        except (TypeError, ValueError): return []
        if movie_id not in self.index: return []
        limit = max(1, int(limit))
        idx = self.index[movie_id]
        sims = cosine_similarity(self.features[idx], self.features).ravel()
        ranked = np.argsort(-sims)
        results = []
        for i in ranked:
            mid = int(self.movies.iloc[i].movieId)
            if mid == movie_id: continue
            results.append(self._record(
                self.movies.iloc[i],
                "Shares genre features with " + str(self.movies.iloc[idx].title), sims[i]
            ))
            if len(results) >= limit: break
        return results

    def for_genres(self, genres, limit=10, exclude=None):
        chosen, excluded = set(genres or []), set(exclude or [])
        if not chosen: return []
        overlaps = np.zeros(len(self.movies), dtype=np.int16)
        for genre in chosen:
            if genre in self.genre_positions: overlaps[self.genre_positions[genre]] += 1
        valid = np.flatnonzero(overlaps > 0)
        if excluded:
            valid = valid[~np.isin(self.movies.movieId.to_numpy()[valid], list(excluded))]
        scores = self.movies.score.to_numpy()
        order = np.lexsort((scores[valid], overlaps[valid]))[::-1]
        results = []
        for pos in valid[order[:max(1,int(limit))]]:
            matched = sorted(chosen.intersection(str(self.movies.iloc[pos].genres).split("|")))
            results.append(self._record(
                self.movies.iloc[pos], "Matches your preferred " + ", ".join(matched) + " genres"
            ))
        return results

    def search(self, query="", genre="", sort="rating"):
        frame = self.movies
        if query: frame = frame[frame.title.str.contains(re.escape(query), case=False, na=False)]
        if genre: frame = frame[frame.genres.str.split("|").apply(lambda values: genre in values)]
        key, ascending = {"rating":("score",False), "popular":("rating_count",False),
                          "title":("title",True)}.get(sort, ("score",False))
        frame = frame.sort_values(key, ascending=ascending)
        return [self._record(row) for _,row in frame.iterrows()]

    def user_preferences(self, ratings_by_movie):
        totals = {}
        for mid, value in ratings_by_movie.items():
            item = self.movie(mid)
            if not item: continue
            for genre in item["genres"].split(", "):
                totals[genre] = totals.get(genre,0) + (float(value)-2.5)
        return sorted((g for g,v in totals.items() if v>0), key=lambda g:totals[g], reverse=True)[:5]

engine = Recommender(movies, ratings)
print(f"Ready: {len(engine.movies):,} movies, {len(engine.genres)} genres.")

## 11. Implement the Existing Recommendation Algorithm
Movie similarity uses cosine similarity between cached TF-IDF genre vectors and skips the selected film. Genre recommendations rank by number of genres matched, then the smoothed ratings score. The popular list uses Bayesian smoothing. A rating above 2.5 strengthens its film's genres in the demo preference profile.

In [ ]:
assert len(engine.movies) == len(movies)
assert engine.movie(-999999) is None
assert len(engine.similar(int(movies.iloc[0].movieId), 5)) <= 5
print("Smoke checks passed: engine built, valid similar-movie query works, invalid ID is handled.")

## 12. Recommend Movies Based on User Preferences or Movie Similarity
Edit movie_query and preferred_genres to explore the catalog. An unmatched title gives a clear message and falls back to the first real catalog title for a runnable example.

In [ ]:
def find_movie(query):
    query = str(query or "").strip()
    if not query: return None
    exact = movies[movies.title.str.casefold() == query.casefold()]
    if not exact.empty: return int(exact.iloc[0].movieId)
    found = movies[movies.title.str.contains(re.escape(query), case=False, na=False)]
    return int(found.iloc[0].movieId) if not found.empty else None

movie_query = "Toy Story"
preferred_genres = ["Adventure", "Comedy"]
seed_id = find_movie(movie_query)
if seed_id is None:
    print(f"No title matched {movie_query!r}; using first catalog title for demonstration.")
    seed_id = int(movies.iloc[0].movieId)
seed = engine.movie(seed_id)
similar_recs = engine.similar(seed_id, 8)
genre_recs = engine.for_genres(preferred_genres, 8)
# Demonstrate the existing app's rating-to-genre preference logic with real catalog films.
demo_user_ratings = {
    int(movies.iloc[0].movieId): 5.0,
    int(movies.iloc[min(50, len(movies)-1)].movieId): 4.5,
}
inferred_genres = engine.user_preferences(demo_user_ratings)
personal_recs = engine.for_genres(inferred_genres, 8, exclude=list(demo_user_ratings))
print("Selected movie:", seed["title"], "| Genres:", seed["genres"])
print("Chosen genres:", preferred_genres)
print("Genres inferred from sample user ratings:", inferred_genres)

## 13. Display Recommended Movies in a Clean Table
Results show MovieLens mean rating, number of ratings, recommendation reason, and genre similarity when applicable.

In [ ]:
def recommendation_table(records):
    cols = ["title","genres","rating","rating_count","similarity","reason"]
    if not records: return pd.DataFrame(columns=cols)
    return pd.DataFrame(records)[cols].rename(columns={
        "title":"Movie", "genres":"Genres", "rating":"Mean rating / 5",
        "rating_count":"Rating count", "similarity":"Genre similarity", "reason":"Why recommended"
    })
print("Similar to selected movie")
display(recommendation_table(similar_recs))
print("Chosen genre matches")
display(recommendation_table(genre_recs))
print("Personalized matches from sample ratings")
display(recommendation_table(personal_recs))

## 14. Evaluate the Recommendation System
This preserves the project's deterministic 80/20 rating holdout (random_state 42), relevance threshold of rating ≥ 4, and Precision@K, Recall@K, catalog coverage, MAE, and RMSE. MAE/RMSE assess smoothed item-average rating estimates, not genre similarity. Scores below are calculated from this dataset run.

In [ ]:
def evaluate_recommender(movies, ratings, k=10):
    if len(ratings) < 20:
        return {"precision_at_k":0.0,"recall_at_k":0.0,"mae":None,"rmse":None,
                "catalog_coverage":0.0,"k":k,"users_evaluated":0,
                "note":"At least 20 ratings are needed for a meaningful holdout."}
    train, test = train_test_split(ratings, test_size=0.2, random_state=RANDOM_STATE)
    train_engine = Recommender(movies, train)
    groups = train.groupby("userId")
    hits, relevant_total = 0, 0
    covered, actual, predicted = set(), [], []
    for user, test_group in test.groupby("userId"):
        known = groups.get_group(user) if user in groups.groups else train.iloc[0:0]
        liked = known[known.rating >= 4.0]
        weights = {}
        for mid in liked.movieId:
            item = engine.movie(int(mid))
            if item:
                for genre in item["genres"].split(", "): weights[genre] = weights.get(genre,0)+1
        top_genres = sorted(weights, key=weights.get, reverse=True)[:3]
        recs = (train_engine.for_genres(top_genres,k,exclude=known.movieId.tolist())
                if top_genres else train_engine.popular(k))
        rec_ids = {x["movieId"] for x in recs}
        covered.update(rec_ids)
        truth = set(test_group.loc[test_group.rating >= 4.0, "movieId"].astype(int))
        hits += len(truth & rec_ids)
        relevant_total += len(truth)
        actual.extend(test_group.rating.astype(float).tolist())
        for mid in test_group.movieId:
            item = train_engine.movie(int(mid))
            predicted.append(float(item["rating"]) if item else train_engine.global_mean)
    user_count = max(1, test.userId.nunique())
    return {
        "precision_at_k":round(hits/(user_count*k),4),
        "recall_at_k":round(hits/relevant_total,4) if relevant_total else 0.0,
        "mae":round(float(mean_absolute_error(actual,predicted)),3),
        "rmse":round(float(np.sqrt(mean_squared_error(actual,predicted))),3),
        "catalog_coverage":round(len(covered)/max(1,movies.movieId.nunique()),4),
        "k":k,"users_evaluated":int(user_count),
        "note":"Fixed 80/20 holdout; relevant means rating ≥ 4."
    }

evaluation = evaluate_recommender(movies, ratings, k=10)
display(pd.DataFrame([evaluation]))
print(evaluation["note"])

## 15. Test the System with Sample Inputs
Try another title substring, a MovieLens ID, or a different genre selection. Invalid inputs are handled without crashing.

In [ ]:
sample_id = find_movie("Matrix")
if sample_id is None:
    print("No Matrix title in this catalog version. Try a title from movies.head().")
else:
    print("Matched:", engine.movie(sample_id)["title"])
    display(recommendation_table(engine.similar(sample_id, 5)))
bad_result = engine.similar("not-a-movie-id", 5)
print("Invalid movie ID:", "No recommendations returned (handled safely)." if not bad_result else "Unexpected result")

## 16. Results and Observations
The summary reports actual counts, observed mean, and holdout metrics from this run. Dataset versions may change. Genre overlap is explainable, but genre labels alone cannot capture plot, tone, or audience.

In [ ]:
print(f"Data: {len(movies):,} movies, {len(ratings):,} ratings, {ratings.userId.nunique():,} users")
print(f"Observed mean rating: {ratings.rating.mean():.2f}/5")
print(f"Precision@{evaluation['k']}: {evaluation['precision_at_k']:.4f}")
print(f"Recall@{evaluation['k']}: {evaluation['recall_at_k']:.4f}")
print(f"Catalog coverage: {evaluation['catalog_coverage']:.1%}")
print(f"Smoothed item-rating estimate MAE/RMSE: {evaluation['mae']} / {evaluation['rmse']}")

## 17. Conclusion
The Colab notebook retains MovieMind's real recommendation behavior: genre TF-IDF/cosine similarity, genre matching, preference weighting, and smoothed popularity. Charts, recommendations, and metrics are generated from MovieLens CSVs at runtime. This is a lightweight, explainable content-based system using rating aggregates rather than latent collaborative filtering.

## 18. Future Improvements
- Add and fairly compare collaborative filtering on user–movie interactions.
- Enrich genre features with tags or descriptions.
- Compare ranking cutoffs and time-aware holdouts.
- Persist user profiles for a multi-user application.
- Explore hybrid ranking and cold-start strategies while preventing evaluation leakage.